# FPP07 — Minutiae Local Structure: Neighbourhood Size, then Neighbour Attention (incremental over FPP06; 3 repetitions)

**Question:** FPP06 chose the **local-structure** encoder (9.99 % source-balanced validation EER, k = 16 neighbours, max over neighbours). Does the local structure improve with (A) **more or fewer neighbours**, then (B) **attention over the neighbours** instead of max?

**Incremental, nothing repeated:** the **baseline** is the FPP06 winner, **re-scored from its saved outputs, not re-trained** (row `k16 (FPP06)`). Every run **starts from the FPP06 weights of the same repetition** — k does not change any weight shape, so all weights transfer; the attention arm adds only one score layer per neighbour layer (new, small). Same augmentation, data, loss and sampler as FPP06; only the neighbourhood changes.

| Step | Arms | Start weights | Kept if |
|---|---|---|---|
| A — neighbourhood size | `k8`, `k24`, `k32` (baseline `k16`) | FPP06 winner | lowest balanced validation EER among k8 / k16 / k24 / k32 |
| B — neighbour aggregation | `k*_attention`: a learned score per neighbour edge, softmax over the k* neighbours, weighted sum (replaces the max) | step-A winner | it beats the step-A winner |

**Training (fine-tuning from FPP06):** AdamW **3e-4** (FPP06 trained from scratch at 1e-3 and ended after LR drops), weight decay 1e-4, clip 1.0, ≤ 60 epochs, early stopping / LR drops / best epoch on the **source-balanced validation EER** (patience 10, LR × 0.3 after 3 — the fine-tuning rule of FPP04C / FPP05); batch 64 pairs, 12,000 pairs per epoch, source-balanced sampler; FPP06 augmentation (25 % original, else rotation ± 5°, shift ± 10 px, 10 % minutia dropout, 2 px / 5° jitter).

**Decision (validation only):** final winner = lowest mean source-balanced validation EER among `k16 (FPP06)`, the step-A arms and the step-B arm (AUC breaks ties). Validation table: balanced + DB1 / DB2 / DB1×DB2 — EER, AUC, F1, FMR, FNMR (mean ± std) — with **FingerJetFX + BOZORTH3** and the **image model** as reference rows. **Test and FingerDOS are reported for every row, never used to decide.** Bias check with the 5 calibration subjects; rotation check per run (must stay ≈ 1).

**Time:** more neighbours cost more compute — GPU per run and the fair CPU time (1 thread, fresh process: FingerJetFX + set + embedding + cosine vs FingerJetFX + BOZORTH3) for every candidate.

**Hardware:** A100; 4 arms × 3 repetitions ≈ 3–4 h (split with `ARMS_TO_RUN`). **Needs:** FPP06 finished (winner = local_structure), FPP01–FPP05 outputs as in FPP06, fpp_lib ≥ 1.7.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 7): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.7')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration and the FPP06 winner

In [ ]:
import h5py, math, subprocess, tensorflow as tf
import matplotlib.pyplot as plt
KERNEL = 'FPP07'; PREV_KERNEL = 'FPP06'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'
REPS = L.repetitions(ws); REPS_TO_RUN = REPS
STEP_A = ['k8', 'k24', 'k32']          # step B arm = f'k{K}_attention' at the step-A winner's k
BASE = 'k16 (FPP06)'                   # baseline label (FPP06 winner, re-scored)
ARMS_TO_RUN = STEP_A                   # e.g. ['k8'] to split step A across sessions
LR = 3e-4; WEIGHT_DECAY = 1e-4; CLIPNORM = 1.0; EPOCHS = 60; PATIENCE = 10; LR_PATIENCE = 3; LR_FACTOR = 0.3
PAIR_BATCH = 64; TRAIN_PAIRS_PER_EPOCH = 12000; VAL_PAIRS = None; EMBED_BATCH = 512; TIMING_REPEATS = 30
CPU_TIMING_TEMPLATES = 300; CPU_TIMING_WARMUP = 20; ROTATION_CHECK_TEMPLATES = 256
TRAIN_AUGMENTATION = True              # the same augmentation for both arms (as FPP04 used the FPP01 augmentation for every backbone)
QUICK_SMOKE_TEST = False
if QUICK_SMOKE_TEST:
    EPOCHS = 2; TRAIN_PAIRS_PER_EPOCH = 384; VAL_PAIRS = 384; REPS_TO_RUN = REPS[:1]; ARMS_TO_RUN = ['k8']; KDIR = KDIR / '_smoke_test'; FIG = KDIR / 'figures'; CPU_TIMING_TEMPLATES = 30
FIG.mkdir(parents=True, exist_ok=True); (KDIR / 'runs').mkdir(parents=True, exist_ok=True); LOG = L.RunLog(KDIR / 'FPP07_run_log.csv')
tf.keras.mixed_precision.set_global_policy('float32')
# the encoder / set code lives in one module: the notebook and the 1-thread CPU timing worker import the same file (also saved next to the results)
MOD_PATH = Path('/content/fpp07_minutiae.py'); MOD_PATH.write_text(r'''# fpp07_minutiae.py — the FPP06 module + neighbour aggregation option ('max' = FPP06, 'attention' = learned weights over the k neighbours)
import numpy as np
SETUP = dict(quality=False, k=16, depth=2, N=128, aggregate='max')          # architecture ablation: pilot defaults (FPM07), no quality input
ST = dict(d=256, heads=4, blocks=3, ff=512)                 # Set Transformer: model width, heads, self-attention blocks, feed-forward width
EDGE1, EDGE2, POINT = [64, 128], [256, 256], 512; EMBEDDING_DIM = 1024; DROPOUT = 0.3; GEO_SCALE = 4.0
AUG = dict(rotation_deg=5.0, shift_px=10.0, dropout=0.10, position_sigma_px=2.0, angle_sigma_deg=5.0, p_original=0.25)
# placement (as the FPP01 image recipe: rotation +-5 deg, shift +-3 % of the image ~ +-10 px at 500 ppi) + FingerJetFX noise (FPM08: 10 % minutia dropout, 2 px / 5 deg jitter); 25 % of draws stay original (as FPP01 mix_versions)
COORD_HALF = 112.0      # native 500 ppi pixels, centred on the image, / 112 (one physical scale for every sensor)

def load_minutiae(h5file):
    import h5py
    out = []
    with h5py.File(h5file, 'r') as h:
        g = h['minutiae']; X, Y, TH, TY, Q = g['x'][:], g['y'][:], g['theta'][:], g['type'][:], g['quality'][:]; W, H = g['image_width'][:], g['image_height'][:]
        PPI = g['ppi'][:] if 'ppi' in g else np.full(len(X), 500)
    for i in range(len(X)):
        f = 500.0 / float(PPI[i] or 500)
        out.append({'x': ((np.asarray(X[i], np.float32) - W[i] / 2) * f).astype(np.float32), 'y': ((np.asarray(Y[i], np.float32) - H[i] / 2) * f).astype(np.float32),
                    'theta': np.deg2rad(np.asarray(TH[i], np.float32)).astype(np.float32), 'type': np.clip(np.asarray(TY[i], np.int64), 0, 2),
                    'quality': np.asarray(Q[i], np.float32)})
    return out

def augment(m, rng):
    # training only: with p 0.25 the original; else whole-print rotation + shift (placement), then minutia dropout and position / angle jitter (extraction noise)
    if len(m['x']) == 0 or rng.random() < AUG['p_original']: return m
    m = rotate_translate(m, np.deg2rad(rng.uniform(-AUG['rotation_deg'], AUG['rotation_deg'])), rng.uniform(-AUG['shift_px'], AUG['shift_px']), rng.uniform(-AUG['shift_px'], AUG['shift_px']))
    x, y, t, ty, q = m['x'].copy(), m['y'].copy(), m['theta'].copy(), m['type'].copy(), m['quality'].copy()
    keep = rng.random(len(x)) >= AUG['dropout']
    if keep.sum() >= 5: x, y, t, ty, q = x[keep], y[keep], t[keep], ty[keep], q[keep]
    k = len(x); x = x + rng.normal(0, AUG['position_sigma_px'], k).astype(np.float32); y = y + rng.normal(0, AUG['position_sigma_px'], k).astype(np.float32)
    t = t + rng.normal(0, np.deg2rad(AUG['angle_sigma_deg']), k).astype(np.float32)
    return {'x': x.astype(np.float32), 'y': y.astype(np.float32), 'theta': t.astype(np.float32), 'type': ty, 'quality': q}

def to_set(m, n_max=128, quality=False):
    # (n_max, F + 1): x, y (/ COORD_HALF), cos, sin, type one-hot (3), [quality], mask; extra minutiae dropped by lowest quality
    F_ = 8 if quality else 7; n = len(m['x']); keep = np.argsort(-m['quality'], kind='stable')[:n_max] if n > n_max else np.arange(n); k = len(keep)
    s = np.zeros((n_max, F_ + 1), np.float32)
    s[:k, 0] = m['x'][keep] / COORD_HALF; s[:k, 1] = m['y'][keep] / COORD_HALF; s[:k, 2] = np.cos(m['theta'][keep]); s[:k, 3] = np.sin(m['theta'][keep])
    s[np.arange(k), 4 + m['type'][keep]] = 1.0
    if quality: s[:k, 7] = m['quality'][keep]
    s[:k, F_] = 1.0; return s

def rotate_translate(m, angle, tx, ty):
    c, s = np.cos(angle), np.sin(angle); x, y = m['x'], m['y']
    return {**m, 'x': (c * x - s * y + tx).astype(np.float32), 'y': (s * x + c * y + ty).astype(np.float32), 'theta': (m['theta'] + angle).astype(np.float32)}

def build_ls_encoder(cfg=SETUP, emb_dim=EMBEDDING_DIM):
    # local structure (FPM07 / FPM08): each minutia described by its k nearest neighbours in its own frame -> invariant to rotation and translation by design
    import tensorflow as tf
    Lyr = tf.keras.layers
    class MaskedPool(Lyr.Layer):
        def call(self, inputs):
            h, m = inputs; m3 = tf.cast(m, h.dtype)[..., None]; cnt = tf.reduce_sum(m3, 1)
            mean = tf.reduce_sum(h * m3, 1) / tf.maximum(cnt, 1.0); mx = tf.reduce_max(h + (m3 - 1.0) * 1e9, 1)
            return tf.concat([tf.where(cnt > 0, mx, tf.zeros_like(mx)), mean], -1)
    class KNN(Lyr.Layer):
        def __init__(self, k, **kw): super().__init__(**kw); self.k = int(k)
        def call(self, inputs):
            x, m = inputs; px, py = x[..., 0], x[..., 1]; n = tf.shape(x)[1]
            d2 = (px[:, None, :] - px[:, :, None]) ** 2 + (py[:, None, :] - py[:, :, None]) ** 2
            d2 = d2 + (1.0 - m[:, None, :]) * 1e9 + tf.eye(n, dtype=d2.dtype)[None] * 2e9
            return tf.math.top_k(-d2, k=self.k).indices
    class EdgeGeometry(Lyr.Layer):
        def __init__(self, quality=False, **kw): super().__init__(**kw); self.quality = bool(quality)
        def call(self, inputs):
            x, m, idx = inputs; xj = tf.gather(x, idx, batch_dims=1); xi = x[:, :, None, :]
            dx = xj[..., 0] - xi[..., 0]; dy = xj[..., 1] - xi[..., 1]; ci, si, cj, sj = xi[..., 2], xi[..., 3], xj[..., 2], xj[..., 3]
            u = dx * ci + dy * si; v = -dx * si + dy * ci; dist = tf.sqrt(u * u + v * v + 1e-12)
            ti = tf.broadcast_to(xi[..., 4:7], tf.shape(xj[..., 4:7])); valid = tf.gather(m, idx, batch_dims=1) * m[:, :, None]
            parts = [tf.stack([GEO_SCALE * u, GEO_SCALE * v, GEO_SCALE * dist, cj * ci + sj * si, sj * ci - cj * si], -1), ti, xj[..., 4:7]]
            if self.quality: parts += [tf.broadcast_to(xi[..., 7:8], tf.shape(xj[..., 7:8])), xj[..., 7:8]]
            return tf.concat(parts + [valid[..., None]], -1)
    class MaskedAttnK(Lyr.Layer):
        # neighbour attention: softmax over the k neighbours of a learned score per edge (padding masked); weighted sum of the edge features
        def call(self, inputs):
            e, s, valid = inputs; logits = s[..., 0] + (valid - 1.0) * 1e9; w = tf.nn.softmax(logits, axis=2)
            out = tf.reduce_sum(w[..., None] * e, 2); return tf.where(tf.reduce_sum(valid, 2, keepdims=True) > 0, out, tf.zeros_like(out))
    def aggregate(e, valid, name):
        if cfg.get('aggregate', 'max') == 'max': return MaskedMaxK(name=name)([e, valid])
        return MaskedAttnK(name=name)([e, Lyr.Dense(1, name=f'{name}_attn_score')(e), valid])
    class GatherNeighbours(Lyr.Layer):
        def call(self, inputs): h, idx = inputs; return tf.gather(h, idx, batch_dims=1)
    class MaskedMaxK(Lyr.Layer):
        def call(self, inputs):
            e, valid = inputs; v = valid[..., None]; mx = tf.reduce_max(e + (v - 1.0) * 1e9, 2)
            return tf.where(tf.reduce_sum(v, 2) > 0, mx, tf.zeros_like(mx))
    F_ = 8 if cfg['quality'] else 7; k = cfg['k']; inp = tf.keras.Input((cfg['N'], F_ + 1), dtype=tf.float32, name='minutiae_set')
    x = Lyr.Lambda(lambda z: z[..., :F_], name='features')(inp); m = Lyr.Lambda(lambda z: z[..., F_], name='mask')(inp)
    idx = KNN(k, name='knn')([x, m]); eg = EdgeGeometry(cfg['quality'], name='edge_geometry')([x, m, idx])
    geo = Lyr.Lambda(lambda z: z[..., :-1], name='edge_features')(eg); valid = Lyr.Lambda(lambda z: z[..., -1], name='edge_valid')(eg)
    e = geo
    for i, u in enumerate(EDGE1): e = Lyr.Dense(u, activation='relu', name=f'edge1_dense_{i}')(e)   # explicit names: weights load by structure in any process
    h = aggregate(e, valid, 'local_structure_1'); hs = [h]
    for layer in range(2, cfg['depth'] + 1):
        hj = GatherNeighbours(name=f'gather_{layer}')([h, idx]); hi = Lyr.Lambda(lambda z: tf.repeat(z[:, :, None, :], k, axis=2), name=f'tile_{layer}')(h)
        e2 = Lyr.Concatenate(name=f'edge_concat_{layer}')([hi, Lyr.Subtract(name=f'edge_diff_{layer}')([hj, hi]), geo])
        for i, u in enumerate(EDGE2): e2 = Lyr.Dense(u, activation='relu', name=f'edge{layer}_dense_{i}')(e2)
        h = aggregate(e2, valid, f'local_structure_{layer}'); hs.append(h)
    h = Lyr.Dense(POINT, activation='relu', name='point_dense')(Lyr.Concatenate(name='local_concat')(hs) if len(hs) > 1 else hs[0]); pooled = MaskedPool(name='pool')([h, m])
    f = Lyr.Dropout(DROPOUT, name='head_dropout')(pooled); e = Lyr.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='embedding')(e)
    return tf.keras.Model(inp, e, name='minutiae_local_structure_encoder')

def build_st_encoder(cfg=SETUP, st=ST, emb_dim=EMBEDDING_DIM):
    # Set Transformer (Lee et al. 2019): per-minutia features -> pre-norm self-attention blocks over all minutiae (padding masked) -> attention pooling (PMA, 1 seed) + masked mean
    import tensorflow as tf
    Lyr = tf.keras.layers
    class Seed(Lyr.Layer):
        def build(self, shape): self.s = self.add_weight(name='seed', shape=(1, 1, st['d']), initializer='glorot_uniform')
        def call(self, h): return tf.tile(self.s, [tf.shape(h)[0], 1, 1])
    class MaskedMean(Lyr.Layer):
        def call(self, inputs):
            h, m = inputs; m3 = m[..., None]; return tf.reduce_sum(h * m3, 1) / tf.maximum(tf.reduce_sum(m3, 1), 1.0)
    F_ = 8 if cfg['quality'] else 7; inp = tf.keras.Input((cfg['N'], F_ + 1), dtype=tf.float32, name='minutiae_set')
    x = Lyr.Lambda(lambda z: z[..., :F_], name='features')(inp); m = Lyr.Lambda(lambda z: z[..., F_], name='mask')(inp)
    key_mask = Lyr.Lambda(lambda z: tf.tile(z[:, None, :] > 0.5, [1, tf.shape(z)[1], 1]), name='key_mask')(m)       # (B, N, N): attend to real minutiae only
    seed_mask = Lyr.Lambda(lambda z: z[:, None, :] > 0.5, name='seed_mask')(m)                                       # (B, 1, N)
    h = Lyr.Dense(st['d'], name='input_proj')(x)
    for b in range(st['blocks']):
        a = Lyr.LayerNormalization(name=f'sab{b}_ln1')(h)
        a = Lyr.MultiHeadAttention(st['heads'], st['d'] // st['heads'], name=f'sab{b}_mha')(a, a, attention_mask=key_mask)
        h = Lyr.Add(name=f'sab{b}_res1')([h, a])
        f = Lyr.LayerNormalization(name=f'sab{b}_ln2')(h); f = Lyr.Dense(st['ff'], activation='gelu', name=f'sab{b}_ff1')(f); f = Lyr.Dense(st['d'], name=f'sab{b}_ff2')(f)
        h = Lyr.Add(name=f'sab{b}_res2')([h, f])
    h = Lyr.LayerNormalization(name='final_ln')(h)
    q = Seed(name='pma_seed')(h); pma = Lyr.MultiHeadAttention(st['heads'], st['d'] // st['heads'], name='pma_mha')(q, h, attention_mask=seed_mask)
    pooled = Lyr.Concatenate(name='pool')([Lyr.Flatten(name='pma_flat')(pma), MaskedMean(name='masked_mean')([h, m])])
    f = Lyr.Dropout(DROPOUT, name='head_dropout')(pooled); e = Lyr.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='embedding')(e)
    return tf.keras.Model(inp, e, name='minutiae_set_transformer_encoder')

BUILDERS = {'set_transformer': build_st_encoder, 'local_structure': build_ls_encoder}
def build_encoder(arch): return BUILDERS[arch]()

def arm_cfg(arm):
    # 'k16' (FPP06) / 'k8' / 'k24' / 'k32' / 'k<K>_attention'
    k = int(arm.split('_')[0][1:]); return dict(SETUP, k=k, aggregate='attention' if arm.endswith('_attention') else 'max')
def build_arm(arm): return build_ls_encoder(arm_cfg(arm))
'''); sys.path.insert(0, '/content')
import fpp07_minutiae as M; importlib.reload(M)
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], protocol=P.PROTOCOL_TABLE, step_a=STEP_A, step_b='k<step-A winner>_attention', baseline=f'{BASE}: FPP06 winner re-scored, not re-trained', start='FPP06 winner weights of the same repetition (step A); step-A winner (step B)', setup=M.SETUP,
              local_structure=dict(k=M.SETUP['k'], depth=M.SETUP['depth'], edge1=M.EDGE1, edge2=M.EDGE2, point=M.POINT, geometry_scale=M.GEO_SCALE),
              embedding_dim=M.EMBEDDING_DIM, dropout=M.DROPOUT, coordinates='native 500 ppi, centred, / 112', quality_input=False, augmentation=(M.AUG if TRAIN_AUGMENTATION else None),
              loss='cosine-BCE (learned calibration)', lr=LR, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM, epochs=EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE,
              lr_factor=LR_FACTOR, pair_batch=PAIR_BATCH, train_pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH, sampler='source-balanced DB1 / DB2 / DB1xDB2',
              early_stopping=P.SELECTION_RULE, threshold=P.THRESHOLD_RULE, decision=P.DECISION_RULE, local_runs=str(P.LOCAL_RUNS),
              repetitions=REPS, quick_smoke_test=QUICK_SMOKE_TEST, fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))
# ---- the FPP06 winner (start weights and baseline; never re-trained)
mf6 = json.loads((P.kernel_dir(ws, PREV_KERNEL) / 'winner_manifest.json').read_text())
if mf6['winning_arm'] != 'local_structure': raise RuntimeError(f"FPP07 refines the local structure, but FPP06 chose {mf6['winning_arm']}")
BASE_DIR = P.kernel_dir(ws, PREV_KERNEL); BASE_ARM = mf6['winning_arm']
def run_dir(a, r): return BASE_DIR / 'runs' / r / BASE_ARM if a == BASE else KDIR / 'runs' / r / a
def cfg_of(a): return M.arm_cfg('k16' if a == BASE else a)
print('baseline / start:', BASE_DIR.name, '/', BASE_ARM, '| FPP06 decision:', [(d['kernel'], round(d['eer'], 2)) for d in mf6['decision']])

## 2. FingerJetFX templates → minutiae sets, protocol, augmented pair sequence (as FPP06)

In [ ]:
MINU = {d: M.load_minutiae(L.local_copy(ws['templates'] / f'{d}_fingerjetfx_templates.h5')) for d in L.DATASETS}
SETS = {d: np.stack([M.to_set(m) for m in MINU[d]]) for d in L.DATASETS}
for d in L.DATASETS:
    cnt = np.array([len(m['x']) for m in MINU[d]])
    print(f"{d}: {len(cnt)} templates | minutiae median {int(np.median(cnt))}, max {cnt.max()}, empty {int((cnt == 0).sum())}, > N={M.SETUP['N']}: {100 * (cnt > M.SETUP['N']).mean():.1f} %")
MOLF = ('MOLF_DB1', 'MOLF_DB2'); MINU_MOLF = MINU['MOLF_DB1'] + MINU['MOLF_DB2']; SETS_MOLF = np.concatenate([SETS[d] for d in MOLF])
OFFSET = {'MOLF_DB1': 0, 'MOLF_DB2': len(MINU['MOLF_DB1'])}
IMG_MOLF = SETS_MOLF          # the validation callback embeds INPUT[bv.indices]; here the inputs are minutiae sets
PROTO = {r: P.load_pairs(ws, r) for r in REPS}; print('FPP01 pair files verified for', len(REPS), 'repetitions')

class SetPairSeq(tf.keras.utils.Sequence):
    # source-balanced pairs of minutiae sets; each training template freshly augmented (M.augment), reproducible per (seed, epoch, batch); the same for both arms
    def __init__(self, sampler, seed, workers=4):
        super().__init__(workers=workers, use_multiprocessing=False, max_queue_size=8); self.s = sampler; self.seed = seed; self.epoch = 0
    def __len__(self): return len(self.s)
    def _sets(self, idx, rng): return np.stack([M.to_set(M.augment(MINU_MOLF[int(i)], rng)) for i in idx]) if TRAIN_AUGMENTATION else SETS_MOLF[np.asarray(idx)]
    def __getitem__(self, b):
        ia, ib, y = self.s.batch(b); rng = np.random.default_rng([self.seed, self.epoch, b]); return (self._sets(ia, rng), self._sets(ib, rng)), y
    def on_epoch_end(self): self.s.on_epoch_end(); self.epoch += 1

_r = np.random.default_rng(0); _m = next(mm for mm in MINU['MOLF_DB1'] if len(mm['x']) > 30); _a = [M.augment(_m, _r) for _ in range(200)]
print(f"augmentation check: {np.mean([len(a['x']) for a in _a]) / len(_m['x']):.3f} of minutiae kept on average, {np.mean([a is _m for a in _a]):.2f} of draws original")

fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, d in zip(axs, L.DATASETS):
    m = next(mm for mm in MINU[d] if len(mm['x']) > 20)
    ax.quiver(m['x'], m['y'], np.cos(m['theta']), np.sin(m['theta']), color=[P.COLORS[int(t)] for t in m['type']], scale=25, width=0.004)
    ax.set(aspect='equal', title=f'{d}: {len(m["x"])} minutiae'); ax.invert_yaxis(); ax.grid(alpha=.2)
fig.suptitle('FingerJetFX minutiae in the model frame (native 500 ppi, centred)', fontsize=10); P._save(fig, FIG / 'minutiae_frames.png')

## 3. Siamese model, balanced-EER callback, rotation check, weight transfer

In [ ]:
class EpochClock(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['epoch_seconds'] = time.perf_counter() - self.t0
class BalancedEERCallback(tf.keras.callbacks.Callback):
    # after each epoch: embed the validation images, score all validation pairs, source-balanced EER -> best checkpoint / LR drop / early stop (fpp_lib rule)
    def __init__(self, encoder, bv, ckpt, hist_csv):
        super().__init__(); self.encoder = encoder; self.bv = bv; self.ckpt = ckpt; self.hist_csv = hist_csv; self.x = IMG_MOLF[bv.indices]; self.rows = []
        self.stop = P.BalancedEERStopping(PATIENCE, LR_PATIENCE, LR_FACTOR)
    def _lr(self): return float(np.asarray(self.model.optimizer.learning_rate))
    def _set_lr(self, v):
        try: self.model.optimizer.learning_rate.assign(v)
        except AttributeError: self.model.optimizer.learning_rate = v
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}; t0 = time.perf_counter()
        m = self.bv.score(L.l2n(T.embed(self.encoder, self.x, EMBED_BATCH))); logs.update(m); lr = self._lr(); d = self.stop.update(epoch, m['val_balanced_eer'], m['val_balanced_auc'])
        if d['improved']: self.model.save_weights(str(self.ckpt))
        if d['reduce_lr'] and lr > self.stop.min_lr: self._set_lr(max(lr * LR_FACTOR, self.stop.min_lr))
        if d['stop']: self.model.stop_training = True
        logs['validation_seconds'] = time.perf_counter() - t0
        self.rows.append({'epoch': epoch + 1, **{k: float(np.asarray(v)) for k, v in logs.items() if np.asarray(v).size == 1}, 'lr': lr, 'improved': d['improved']}); pd.DataFrame(self.rows).to_csv(self.hist_csv, index=False)
        print(f"  epoch {epoch + 1}: val balanced EER {100 * m['val_balanced_eer']:.2f} % | AUC {100 * m['val_balanced_auc']:.2f} % | "
              + ' '.join(f"{n.split('_')[0]} {100 * m[f'val_eer_{n}']:.2f}" for n in P.VAL_SOURCES) + (' *best' if d['improved'] else '') + (' | LR down' if d['reduce_lr'] else '') + (' | stop' if d['stop'] else ''))

class SetSiamese(tf.keras.Model):
    # shared encoder, cosine-BCE with a learned calibration (as every FPP kernel)
    def __init__(self, encoder, **kw):
        super().__init__(**kw); self.encoder = encoder; self.calib = T.CosineCalibration(name='same_finger_probability'); self.bce = tf.keras.losses.BinaryCrossentropy()
        self.t_loss = tf.keras.metrics.Mean(name='loss'); self.t_acc = tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'); self.t_auc = tf.keras.metrics.AUC(name='verification_auc')
    @property
    def metrics(self): return [self.t_loss, self.t_acc, self.t_auc]
    def call(self, inputs, training=False):
        ea = self.encoder(inputs[0], training=training); eb = self.encoder(inputs[1], training=training)
        return self.calib(tf.reduce_sum(ea * eb, axis=1))
    def train_step(self, data):
        (xa, xb), y = data; y = tf.cast(tf.reshape(y, [-1]), tf.float32)
        with tf.GradientTape() as tape:
            p = self(( xa, xb), training=True); loss = self.bce(y, p)
        tv = self.trainable_variables; self.optimizer.apply_gradients([(g_, v_) for g_, v_ in zip(tape.gradient(loss, tv), tv) if g_ is not None])
        self.t_loss.update_state(loss); self.t_acc.update_state(y, p); self.t_auc.update_state(y, p); return {m_.name: m_.result() for m_ in self.metrics}

def build_set_siamese(encoder, lr):
    m = SetSiamese(encoder, name='set_siamese'); d = np.zeros((1,) + SETS_MOLF.shape[1:], np.float32); m((d, d)); m.built = True
    m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM)); return m

ROT_IDS = [i for i in range(len(MINU['MOLF_DB1'])) if len(MINU['MOLF_DB1'][i]['x']) >= 10][:ROTATION_CHECK_TEMPLATES]
_rr = np.random.default_rng(1); ROT_PARAMS = [(_rr.uniform(-np.pi, np.pi), _rr.uniform(-20, 20), _rr.uniform(-20, 20)) for _ in ROT_IDS]
ROT_A = np.stack([M.to_set(MINU['MOLF_DB1'][i]) for i in ROT_IDS]); ROT_B = np.stack([M.to_set(M.rotate_translate(MINU['MOLF_DB1'][i], *p_)) for i, p_ in zip(ROT_IDS, ROT_PARAMS)])
def rotation_check(encoder):
    # cosine(embedding of a template, embedding of the same template rotated +-180 deg and shifted +-20 px): 1 = fully invariant
    e0 = L.l2n(T.embed(encoder, ROT_A, EMBED_BATCH)); e1 = L.l2n(T.embed(encoder, ROT_B, EMBED_BATCH)); c_ = np.sum(e0 * e1, 1)
    return {'rotation_cos_mean': float(np.mean(c_)), 'rotation_cos_min': float(np.min(c_))}

for a in [BASE] + STEP_A + ['k16_attention']:     # untrained models: finite, unit norm, rotation + translation invariant
    enc = M.build_ls_encoder(cfg_of(a)); e = enc(tf.constant(ROT_A[:8]), training=False).numpy(); rc = rotation_check(enc)
    if not (np.isfinite(e).all() and np.allclose(np.linalg.norm(e, axis=1), 1, atol=1e-3)) or rc['rotation_cos_min'] < 0.999: raise RuntimeError(f'{a}: check failed {rc}')
    print(f"{a}: {enc.count_params():,} parameters | rotation check min cos {rc['rotation_cos_min']:.5f}"); del enc

def load_start(encoder, src_arm, rep):
    # build the start model, load its saved weights, check it reproduces its saved embeddings, copy every weight by layer name into `encoder`
    sd = run_dir(src_arm, rep)
    if not L.run_complete(sd): raise FileNotFoundError(f'{sd} not finished')
    src = M.build_ls_encoder(cfg_of(src_arm)); src.load_weights(str(sd / 'encoder.weights.h5'))
    E0 = L.load_npz(sd / 'embeddings_float32.npz')['MOLF_DB1'][:32]; E1 = L.l2n(T.embed(src, SETS['MOLF_DB1'][:32], 32)); cs = float(np.min(np.sum(L.l2n(E0) * E1, 1)))
    if cs < 0.999: raise RuntimeError(f'{src_arm} {rep}: reloaded start weights do not reproduce the saved embeddings (min cosine {cs:.5f})')
    dst = {l.name: l for l in encoder.layers}; copied = new = 0
    for l in src.layers:
        if not l.weights: continue
        if l.name not in dst or [w.shape for w in dst[l.name].weights] != [w.shape for w in l.weights]: raise RuntimeError(f'cannot transfer layer {l.name}')
        dst[l.name].set_weights(l.get_weights()); copied += 1
    new = sum(1 for l in encoder.layers if l.weights and l.name not in {s.name for s in src.layers})
    del src; return {'start_from': L.rel(ws, sd), 'start_check_cosine': round(cs, 6), 'layers_copied': copied, 'layers_new': new}

## 4. One run = one (arm, repetition), started from the given weights

In [ ]:
def train_run(arm, rep, start_arm):
    rd = KDIR / 'runs' / rep / arm
    if L.run_complete(rd): print(arm, rep, 'complete -> skipped'); return
    lrd = P.local_run_dir(KERNEL + ('_smoke' if QUICK_SMOKE_TEST else ''), rep, arm)
    proto, Pp = PROTO[rep]; seed = proto['seed']; tf.keras.backend.clear_session(); gc.collect(); T.set_all_seeds(seed); t = L.timer()
    encoder = M.build_ls_encoder(cfg_of(arm)); start = load_start(encoder, start_arm, rep); params = T.count_params(encoder); siamese = build_set_siamese(encoder, LR)
    ssf = float(proto['manifest']['same_subject_impostor_fraction'])
    train_seq = SetPairSeq(P.MixedPairs(P.sources(Pp, P.TRAIN_SOURCES, OFFSET), PAIR_BATCH, True, seed, TRAIN_PAIRS_PER_EPOCH, ssf), seed)
    bv = P.BalancedValidation(Pp, OFFSET, max_pairs=VAL_PAIRS)
    LOG.add(step='train_start', arm=arm, repetition=rep, learning_rate=LR, **start, train_composition=json.dumps(train_seq.s.composition),
            validation_pairs=json.dumps(bv.composition), **params)
    ckpt = lrd / 'best_siamese.weights.h5'; hist_csv = lrd / 'training_history.csv'; fit_marker = lrd / 'fit_complete.json'
    if ckpt.is_file() and fit_marker.is_file():
        train_s = json.loads(fit_marker.read_text())['training_seconds']; print(arm, rep, 'fit finished earlier in this session -> evaluating the local checkpoint')
    else:
        cbs = [EpochClock(), BalancedEERCallback(encoder, bv, ckpt, hist_csv)]
        siamese.fit(train_seq, epochs=EPOCHS, callbacks=cbs, verbose=2); train_s = t(); fit_marker.write_text(json.dumps({'training_seconds': train_s}))
    siamese.load_weights(ckpt)
    hist = pd.read_csv(hist_csv); b_i, _ = P.BalancedEERStopping.replay(hist, EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR); best = hist.iloc[b_i]
    emb = {d: L.l2n(T.embed(encoder, SETS[d], EMBED_BATCH)) for d in L.DATASETS}                          # clean templates, inference mode
    encoder.save_weights(lrd / 'encoder.weights.h5'); np.savez_compressed(lrd / 'embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in emb.items()})
    rot = rotation_check(encoder)
    gpu_batch_ms, gpu_single_ms = T.time_encoder(encoder, SETS['MOLF_DB1'][:64], TIMING_REPEATS)
    extra = {'architecture': 'local_structure', **cfg_of(arm), 'learning_rate': LR, **start, **rot, 'early_stopping': P.SELECTION_RULE, 'best_epoch': int(best.epoch), 'epochs_run': int(len(hist)),
             'val_balanced_eer_at_best': float(best.val_balanced_eer), 'val_balanced_auc_at_best': float(best.val_balanced_auc), 'train_auc_at_best': float(best.get('verification_auc', np.nan)),
             'training_seconds': train_s, 'seconds_per_epoch': float(hist['epoch_seconds'].mean()) if 'epoch_seconds' in hist else float('nan'), 'pair_batch': PAIR_BATCH,
             'embed_ms_per_template_gpu_batch64': gpu_batch_ms, 'embed_ms_single_template_gpu': gpu_single_ms, **params}
    S = {n: (np.asarray(Pp[n]['label'], int), L.pair_scores(emb[da], emb[db], Pp[n]).astype(np.float64)) for n, (da, db, _) in P.PAIR_SETS.items()}
    rows, thr = P.evaluate(S, KERNEL, arm, rep, extra)
    (lrd / 'encoder_config.json').write_text(json.dumps({'input': 'FingerJetFX minutiae set', 'architecture': 'local_structure', 'arm': arm, 'builder': f'fpp07_minutiae.build_arm({arm!r})', 'setup': cfg_of(arm), 'start': start,
                                                         'local_structure': dict(edge1=M.EDGE1, edge2=M.EDGE2, point=M.POINT, geometry_scale=M.GEO_SCALE),
                                                         'coord_half': M.COORD_HALF, 'embedding_dim': M.EMBEDDING_DIM, 'dropout': M.DROPOUT, 'loss': 'cosine-BCE', 'augmentation': M.AUG if TRAIN_AUGMENTATION else None,
                                                         'learning_rate': LR, 'module': 'fpp07_minutiae.py'}, indent=2))
    P.save_run(lrd, S, rows, thr, {'kernel': KERNEL, 'arm': arm, 'repetition': rep, 'seed': seed, **extra})
    n_files = P.publish_run(lrd, rd, skip=('fit_complete.json', 'best_siamese.weights.h5'))
    P.fig_set_panel(f'{arm} {rep[:6]} (frozen validation-EER threshold)', {st: [(*P.group_scores(S, st), thr)] for st in P.PAPER_SETS}, FIG / 'runs' / rep / f'{arm}_panel.png')
    P.fig_training_balanced(hist, f'{arm} {rep[:6]} — stopping on source-balanced validation EER', FIG / 'runs' / rep / f'{arm}_training.png')
    v = [r_ for r_ in rows if r_['set'] == P.SELECTION_SET][0]
    LOG.add(step='train_done', arm=arm, repetition=rep, status='ok', minutes=round(t() / 60, 1), best_epoch=int(best.epoch), files_to_drive=n_files,
            val_balanced_eer=round(100 * v['eer'], 2), val_balanced_auc=round(100 * v['roc_auc'], 2), val_balanced_f1=round(100 * v['f1_macro'], 2), rotation_cos_mean=round(rot['rotation_cos_mean'], 4))
    del siamese, encoder, emb; gc.collect()

## 5A. Step A — neighbourhood size (start: FPP06 winner)

In [ ]:
for b in ARMS_TO_RUN:
    for rep in REPS_TO_RUN:
        try: train_run(b, rep, BASE)
        except (tf.errors.ResourceExhaustedError, tf.errors.InternalError) as exc:
            LOG.add(step='train_failed', arm=b, repetition=rep, status='gpu_error', error=repr(exc)[:400]); print('GPU error', b, rep, repr(exc)[:200]); tf.keras.backend.clear_session(); gc.collect()
        except Exception as exc:
            LOG.add(step='train_failed', arm=b, repetition=rep, status='error', error=repr(exc)[:400]); print('FAILED', b, rep, exc)

## 5B. Step A decision (validation only) → step B — neighbour attention at the chosen k (start: step-A winner)

In [ ]:
BOZ = 'FingerJetFX + BOZORTH3 (FPP03)'; IMGREF = 'Image model (FPP04L winner)'
def collect(arms):
    rows = []
    for a in arms:
        for r in REPS_TO_RUN:
            if L.run_complete(run_dir(a, r)) and a != BASE: rows.append(pd.read_csv(run_dir(a, r) / 'metrics.csv'))
    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if len(out): out['kernel'] = out['arm']
    return out
ref_base = pd.DataFrame(P.reference_rows(BASE, BASE_DIR, arm=BASE_ARM, reps=REPS_TO_RUN)); ref_base['kernel'] = BASE
done_a = [a for a in STEP_A if all(L.run_complete(run_dir(a, r)) for r in REPS_TO_RUN)]
if len(done_a) < len(STEP_A): print('Step A not finished yet:', [a for a in STEP_A if a not in done_a])
RES_A = pd.concat([ref_base, collect(done_a)], ignore_index=True)
K_WINNER, decision_a = P.decide(RES_A, [BASE] + done_a)
print('STEP A (validation only):'); display(decision_a.round(3)); display(P.validation_table(RES_A, [BASE] + done_a))
K_STAR = 16 if K_WINNER == BASE else int(K_WINNER[1:]); ARM_B = f'k{K_STAR}_attention'
print(f'step-A winner: {K_WINNER} -> step B arm {ARM_B} (start: {K_WINNER})'); CONFIG.update(step_a_winner=K_WINNER, step_b_arm=ARM_B)
if len(done_a) == len(STEP_A) or QUICK_SMOKE_TEST:
    for rep in REPS_TO_RUN:
        try: train_run(ARM_B, rep, K_WINNER)
        except (tf.errors.ResourceExhaustedError, tf.errors.InternalError) as exc:
            LOG.add(step='train_failed', arm=ARM_B, repetition=rep, status='gpu_error', error=repr(exc)[:400]); print('GPU error', ARM_B, rep, repr(exc)[:200]); tf.keras.backend.clear_session(); gc.collect()
        except Exception as exc:
            LOG.add(step='train_failed', arm=ARM_B, repetition=rep, status='error', error=repr(exc)[:400]); print('FAILED', ARM_B, rep, exc)
else: print('finish step A first (all arms, all repetitions)')

## 6. Final decision on VALIDATION only (FPP06 baseline + step A + step B); references; test and FingerDOS reported, not used

In [ ]:
TRAINED = [a for a in STEP_A + [ARM_B] if all(L.run_complete(run_dir(a, r)) for r in REPS_TO_RUN)]
results = collect(TRAINED); complete = [BASE] + TRAINED; BACKBONES = complete
HIST = []
for a in TRAINED:
    for r in REPS_TO_RUN:
        h = pd.read_csv(run_dir(a, r) / 'training_history.csv'); h.insert(0, 'arm', a); h.insert(1, 'repetition', r); HIST.append(h)
histories = pd.concat(HIST, ignore_index=True) if HIST else pd.DataFrame()
FPP03_DIR = P.kernel_dir(ws, 'FPP03'); ref_b = pd.DataFrame(P.reference_rows(BOZ, FPP03_DIR, arm='BOZORTH3', reps=REPS_TO_RUN)); ref_b['kernel'] = BOZ
ref_i = pd.DataFrame(); mf5 = P.kernel_dir(ws, 'FPP05') / 'winner_manifest.json'
if mf5.is_file():
    m5 = json.loads(mf5.read_text()); img_run = Path(ws['output']) / next(iter(m5['files'].values()))['pair_scores.npz']
    ref_i = pd.DataFrame(P.reference_rows(IMGREF, img_run.parents[3], arm=img_run.parent.name, reps=REPS_TO_RUN)); ref_i['kernel'] = IMGREF
ALLR = pd.concat([ref_base, results, ref_b, ref_i], ignore_index=True); SHOW = complete + [BOZ] + ([IMGREF] if len(ref_i) else [])

# ---- 6.1 decision: validation only
WINNER, decision = P.decide(ALLR, complete)
val_table = P.validation_table(ALLR, SHOW)
print('DECISION RULE:', P.DECISION_RULE); display(decision.round(3)); print('Validation (mean ± std, %; BOZORTH3 and the image model = reference rows):'); display(val_table)
print('WINNER (validation only):', WINNER)
gap = decision.set_index('kernel').loc[WINNER, 'eer'] - ALLR[(ALLR.kernel == BOZ) & (ALLR.set == P.SELECTION_SET)].eer.mean() * 100
print(f'winner vs FingerJetFX + BOZORTH3 on balanced validation EER: {gap:+.2f} points')

# ---- 6.2 training / GPU time, size, rotation check (baseline values from its FPP06 run files)
tcols = ['training_seconds', 'best_epoch', 'epochs_run', 'seconds_per_epoch', 'embed_ms_single_template_gpu', 'embed_ms_per_template_gpu_batch64', 'parameters_total', 'rotation_cos_mean', 'rotation_cos_min']
tsrc = pd.concat([pd.read_csv(run_dir(BASE, r) / 'metrics.csv').assign(kernel=BASE) for r in REPS_TO_RUN] + [results], ignore_index=True)
tsrc = tsrc[tsrc.set == P.SELECTION_SET]; timing = tsrc.groupby('kernel')[[c_ for c_ in tcols if c_ in tsrc.columns]].agg(['mean', 'std']).reindex(complete)
timing.columns = [f'{a}_{b}' for a, b in timing.columns]; display(timing.round(3))

# ---- 6.3 test and FingerDOS — reported only
paper_all, snum_all = P.paper_table(ALLR, SHOW, sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test'])
test_table = P.wide_table(snum_all, SHOW, sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro'))
print('Test and FingerDOS (frozen validation-EER threshold; reported, not used for the decision):'); display(test_table)
paired = pd.concat([P.paired_differences(ALLR, a, [BASE, BOZ]) for a in TRAINED], ignore_index=True) if TRAINED else pd.DataFrame()
if len(paired): print('Paired differences vs the FPP06 baseline and BOZORTH3 (points):'); display(paired[paired.set.isin(['MOLF validation (balanced)'] + list(P.VAL_SOURCES)) & paired.metric.isin(['eer', 'roc_auc', 'f1_macro'])].round(3))

## 6b. Bias check of the threshold: 5 calibration hold-out subjects vs validation (MOLF test, balanced)

In [ ]:
CB = []
for b in complete:
    for r in REPS_TO_RUN:
        rd = run_dir(b, r); proto, Pp = PROTO[r]
        c = P.calibration_check(P.load_scores(rd), L.load_npz(rd / 'embeddings_float32.npz'), P.calibration_pairs(proto)); CB.append({'arm': b, 'repetition': r, **c})
calib = pd.DataFrame(CB); cs = calib.drop(columns='repetition').groupby('arm').agg(['mean', 'std'])
calib_table = pd.DataFrame({'backbone': cs.index,
    'threshold validation': [f"{a:.3f} ± {b_:.3f}" for a, b_ in zip(cs[('threshold_validation', 'mean')], cs[('threshold_validation', 'std')])],
    'threshold calibration (5 subj.)': [f"{a:.3f} ± {b_:.3f}" for a, b_ in zip(cs[('threshold_calibration', 'mean')], cs[('threshold_calibration', 'std')])],
    **{f'test {m.upper()} {tag} (%)': (100 * cs[(f'test_{m}_{tag}', 'mean')]).round(2).to_numpy() for m in ('fmr', 'fnmr', 'f1') for tag in ('validation', 'calibration')}}).reset_index(drop=True)
print('Bias check (paper: supplementary): same operating point from 5 never-used subjects -> the validation threshold is not biased by reuse for selection'); display(calib_table)

## 6c. Fair CPU time: 1 thread, fresh process (every candidate, incl. the FPP06 baseline)

In [ ]:
# probe -> decision for 1:1 from a fingerprint IMAGE:  minutiae path = FingerJetFX extraction + set conversion + embedding + cosine;  BOZORTH3 path = FingerJetFX + ISO->xyt + BOZORTH3.
# FingerJetFX and BOZORTH3 times are read from FPP04C/cpu_timing (same runtime type); everything below is measured here, 1 thread, after loading and warm-up.
WORKER = r'''
import os, sys, json, time, numpy as np
mod_dir, wfile, h5file, out, n_t, warm, embfile, cfg = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5]), int(sys.argv[6]), sys.argv[7], json.loads(sys.argv[8])
os.environ['CUDA_VISIBLE_DEVICES'] = ''; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = '1'
import tensorflow as tf
tf.config.threading.set_intra_op_parallelism_threads(1); tf.config.threading.set_inter_op_parallelism_threads(1)
sys.path.insert(0, mod_dir); import fpp07_minutiae as M
enc = M.build_ls_encoder(cfg); enc.load_weights(wfile); f = tf.function(lambda z: enc(z, training=False))
mins = M.load_minutiae(h5file)[:n_t]
E0 = np.load(embfile)['MOLF_DB1'][:16]; E1 = f(tf.constant(np.stack([M.to_set(m) for m in mins[:16]]))).numpy()
check = float(np.min(np.sum(E0 / np.linalg.norm(E0, axis=1, keepdims=True) * E1, 1)))
if check < 0.999: raise SystemExit(f'CPU copy does not reproduce the saved embeddings (min cosine {check:.5f})')
for i in range(warm): f(tf.constant(M.to_set(mins[i % len(mins)])[None])).numpy()
conv, lat = [], []
for m in mins:
    t = time.perf_counter(); s = M.to_set(m)[None]; conv.append(1000 * (time.perf_counter() - t))
    x = tf.constant(s); t = time.perf_counter(); f(x).numpy(); lat.append(1000 * (time.perf_counter() - t))
g = np.random.default_rng(0).normal(size=(10000, M.EMBEDDING_DIM)).astype(np.float32); g /= np.linalg.norm(g, axis=1, keepdims=True); q = g[0].copy()
t = time.perf_counter()
for i in range(20000): float(q @ g[i % 10000])
cmp_ms = 1000 * (time.perf_counter() - t) / 20000
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), '')
json.dump({'templates_timed': len(lat), 'convert_ms_mean': float(np.mean(conv)), 'embed_ms_mean': float(np.mean(lat)), 'embed_ms_median': float(np.median(lat)),
           'embed_ms_std': float(np.std(lat)), 'embed_ms_p95': float(np.percentile(lat, 95)), 'cosine_1to1_ms': cmp_ms, 'check_cosine_vs_saved_embeddings': check, 'cpu': cpu}, open(out, 'w'))
'''
Path('/content/fpp07_cpu_worker.py').write_text(WORKER)
FX = P.kernel_dir(ws, 'FPP04C') / 'cpu_timing' / 'FPP04C_fingerjetfx_bozorth3_timing.csv'
fx = pd.read_csv(FX) if FX.is_file() else pd.DataFrame()
fjfx_ms = float(fx['fingerjetfx_ms_mean'].mean()) if len(fx) else float('nan'); xyt_ms = float(fx['iso_to_xyt_ms_mean'].mean()) if len(fx) else float('nan')
bz_ms = float(fx['bozorth3_ms_per_pair'].mean()) if len(fx) else float('nan'); cpu_ref = str(fx['cpu'].iloc[0]) if len(fx) and 'cpu' in fx else ''
CPU_ROWS = []; h5 = str(L.local_copy(ws['templates'] / 'MOLF_DB1_fingerjetfx_templates.h5'))
for a in complete:
    w = run_dir(a, REPS_TO_RUN[0]) / 'encoder.weights.h5'; out = f"/content/fpp07_cpu_{a.split(' ')[0]}.json"
    r = subprocess.run([sys.executable, '/content/fpp07_cpu_worker.py', '/content', str(w), h5, out, str(CPU_TIMING_TEMPLATES), str(CPU_TIMING_WARMUP), str(w.with_name('embeddings_float32.npz')), json.dumps(cfg_of(a))], capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-3000:]); raise RuntimeError(f'CPU timing failed for {a}')
    d = json.load(open(out)); CPU_ROWS.append({'method': f'FingerJetFX + {a} encoder + cosine', 'template_creation_ms': fjfx_ms + d['convert_ms_mean'] + d['embed_ms_mean'],
                                               'fingerjetfx_ms': fjfx_ms, 'set_conversion_ms': d['convert_ms_mean'], 'embedding_ms': d['embed_ms_mean'], 'embedding_ms_p95': d['embed_ms_p95'],
                                               'comparison_1to1_ms': d['cosine_1to1_ms'], 'probe_to_decision_1to1_ms': fjfx_ms + d['convert_ms_mean'] + d['embed_ms_mean'] + d['cosine_1to1_ms'],
                                               'check_cosine_vs_saved_embeddings': d['check_cosine_vs_saved_embeddings'], 'cpu': d['cpu']})
CPU_ROWS.append({'method': 'FingerJetFX + BOZORTH3', 'template_creation_ms': fjfx_ms + xyt_ms, 'fingerjetfx_ms': fjfx_ms, 'set_conversion_ms': xyt_ms, 'embedding_ms': 0.0, 'embedding_ms_p95': np.nan,
                 'comparison_1to1_ms': bz_ms, 'probe_to_decision_1to1_ms': fjfx_ms + xyt_ms + bz_ms, 'cpu': cpu_ref})
cpu_fair = pd.DataFrame(CPU_ROWS); cpu_fair['threads'] = 1; cpu_fair['same_cpu_model'] = (cpu_fair.cpu == cpu_ref) if cpu_ref else np.nan
for i_, r_ in cpu_fair.iterrows():
    if r_.method != 'FingerJetFX + BOZORTH3':
        n = (r_.template_creation_ms - (fjfx_ms + xyt_ms)) / max(bz_ms - r_.comparison_1to1_ms, 1e-9); cpu_fair.loc[i_, 'break_even_impressions_vs_BOZORTH3'] = n
print('Fair CPU time (1 thread; FingerJetFX / BOZORTH3 from FPP04C cpu_timing):'); display(cpu_fair.round(4))
if cpu_ref and not cpu_fair.same_cpu_model.all(): print('NOTE: CPU model differs from the FPP04C BOZORTH3 timing runtime: re-run the FPP04C timing cells in this runtime for a like-for-like table')

## 7. Figures

In [ ]:
MR = {a: [run_dir(a, r) for r in REPS_TO_RUN] for a in complete}
_bz = [FPP03_DIR / 'runs' / r / 'BOZORTH3' for r in REPS_TO_RUN]
if all((d_ / 'pair_scores.npz').is_file() for d_ in _bz): MR[BOZ] = _bz
P.standard_figures(FIG, MR, sets=P.PAPER_SETS)
P.fig_metric_bars(snum_all, SHOW, FIG / 'metric_bars_validation.png', sets=['MOLF validation (balanced)'], title='Validation (decision set): candidates and references')
P.fig_metric_bars(snum_all, SHOW, FIG / 'metric_bars_test_reported.png', sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], title='Test and FingerDOS (reported, not used for the decision)')
kk = [a for a in [BASE] + STEP_A if a in complete]; ks = [16 if a == BASE else int(a[1:]) for a in kk]; o = np.argsort(ks)
v = ALLR[ALLR.set == P.SELECTION_SET].groupby('kernel').eer.agg(['mean', 'std'])
fig, ax = plt.subplots(figsize=(7, 4)); ax.errorbar(np.array(ks)[o], 100 * v.loc[np.array(kk)[o], 'mean'], yerr=100 * v.loc[np.array(kk)[o], 'std'], marker='o', capsize=4, label='max over neighbours')
if ARM_B in complete: ax.errorbar([K_STAR], [100 * v.loc[ARM_B, 'mean']], yerr=[100 * v.loc[ARM_B, 'std']], marker='s', color='C3', capsize=4, label='attention over neighbours')
ax.axhline(100 * v.loc[BOZ, 'mean'], color='0.4', ls='--', lw=1, label='FingerJetFX + BOZORTH3') if BOZ in v.index else None
ax.set(xlabel='neighbours k', ylabel='validation EER, source-balanced (%)', title='Neighbourhood size and aggregation (mean ± std, 3 repetitions)'); ax.legend(fontsize=8); ax.grid(alpha=.3)
P._save(fig, FIG / 'neighbourhood_ablation.png'); print('figures in', FIG)

## 8. Manifest and workbook

In [ ]:
if not QUICK_SMOKE_TEST and set(STEP_A + [ARM_B]) <= set(TRAINED):
    files = ('encoder.weights.h5', 'encoder_config.json', 'embeddings_float32.npz', 'thresholds.json', 'pair_scores.npz')
    manifest = {'kernel': KERNEL, 'kernel_name': P.FPP_KERNELS[KERNEL], 'winning_arm': WINNER, 'winner_is_baseline': WINNER == BASE, 'architecture': 'local_structure', 'config': cfg_of(WINNER),
                'step_a_winner': K_WINNER, 'step_b_arm': ARM_B, 'learning_rate': LR, 'loss': 'cosine-BCE', 'augmentation': M.AUG,
                'decision_rule': P.DECISION_RULE, 'decision': decision.to_dict('records'), 'decision_step_a': decision_a.to_dict('records'), 'threshold_rule': P.THRESHOLD_RULE,
                'selection_rule': P.SELECTION_RULE, 'balanced_validation_eer_gap_vs_bozorth3_points': float(gap), 'module': 'fpp07_minutiae.py (build_ls_encoder(config))',
                'files': {r: {k: L.rel(ws, run_dir(WINNER, r) / k) for k in files} for r in REPS_TO_RUN}, 'fpp_lib_version': P.VERSION}
    (KDIR / 'winner_manifest.json').write_text(json.dumps(manifest, indent=2, default=str)); (KDIR / 'fpp07_minutiae.py').write_text(MOD_PATH.read_text())
    print('winner_manifest.json written | winner:', WINNER)
SHEETS = {'Decision (validation)': decision, 'Step A decision': decision_a, 'Validation table': val_table, 'Fair CPU time (1 thread)': cpu_fair, 'Time, size, rotation': timing.reset_index(),
          'Test+FingerDOS (reported)': test_table, 'All sets (mean ± std)': paper_all, 'Paired differences': paired, 'Calibration Bias Check': calib_table, 'Calibration (per rep)': calib,
          'Results (per rep)': P.to_percent(ALLR), 'Training history': histories, 'Config': pd.DataFrame([{k: json.dumps(v, default=str) for k, v in CONFIG.items()}]).T.reset_index(),
          'Run log': LOG.frame()}
L.write_workbook(KDIR / 'FPP07_Minutiae_Neighbourhood_Results.xlsx', SHEETS)
for name, df in (('decision', decision), ('decision_step_a', decision_a), ('validation_table', val_table), ('cpu_time_fair', cpu_fair), ('test_fingerdos_reported', test_table),
                 ('paired_differences', paired), ('calibration_bias_check', calib_table), ('results_per_repetition', P.to_percent(ALLR))):
    df.to_csv(KDIR / f'FPP07_{name}.csv', index=False)
print('FPP07 complete | winner on validation:', WINNER)